# Transfer Learning & CNN Architectures

In this notebook we will cover:

1. Why deeper CNNs were needed
2. Evolution of major CNN architectures
3. The idea of Transfer Learning
4. Feature Extraction vs Fine-Tuning
5. Practical Implementation using PyTorch

# Recap: CNN From Scratch

A Convolutional Neural Network consists of:

- Convolution layers (feature extraction)
- Activation functions (ReLU)
- Pooling layers (downsampling)
- Fully connected layers (classification)

Problems when training from scratch:

- Requires large dataset
- Computationally expensive
- Overfitting on small datasets
- Vanishing gradients in deep networks

Question:
If CNNs work, why not just keep stacking more layers?

# Evolution of CNN Architectures (Brief Overview)

## 1. AlexNet (2012)
- 8 layers
- ReLU activation
- Dropout
- GPU training
- Won ImageNet 2012

Impact:
Deep learning became mainstream.

---

## 2. VGGNet (2014)
- 16 / 19 layers
- Only 3×3 convolutions
- Very deep but huge parameters (~138M)

Problem:
Very heavy model.

---

## 3. ResNet (2015)
- Introduced Residual Connections
- Solved vanishing/degradation problem
- Enabled 50, 101, 152+ layers

Key Idea:
Instead of learning H(x), learn F(x) = H(x) - x
Output = F(x) + x

# Why Transfer Learning?

Training from scratch requires:

- Millions of images
- High computation
- Long training time

Observation:
Early CNN layers learn:
- Edges
- Corners
- Textures

These features are universal.

Idea:
Reuse a model trained on a large dataset (ImageNet)
and adapt it to our smaller dataset.

# Types of Transfer Learning

## 1. Feature Extraction
- Freeze all convolution layers (pretrained model like ImageNet)
- Replace final classifier
- Train only last layer

Used when:
- Small dataset
- Similar domain

---

## 2. Fine-Tuning
- Freeze some layers
- Train last few layers
- Adjust features slightly

Used when:
- Moderate dataset size
- Slight domain difference

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),  # Required for pretrained models
    transforms.ToTensor()
])

full_train = torchvision.datasets.CIFAR10(root='./data',
                                        train=True,
                                        download=True,
                                        transform=transform)

full_test = torchvision.datasets.CIFAR10(root='./data',
                                       train=False,
                                       download=True,
                                       transform=transform)

from torch.utils.data import Subset

train_subset = Subset(full_train, range(500))
test_subset = Subset(full_test, range(150))

trainloader = torch.utils.data.DataLoader(train_subset,
                                          batch_size=16,
                                          shuffle=True)

testloader = torch.utils.data.DataLoader(test_subset,
                                         batch_size=16,
                                         shuffle=False)

In [ ]:
model = models.resnet18(pretrained=True) # pretrained=False? -> train from scratch
# resnet18: 18 layers, residual connections
model = model.to(device)
model

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 231MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Each layer has weights (parameters).

`requires_grad`=False means:

* Do NOT compute gradients.

* Do NOT update these weights.

* So we freeze feature extractor.

This is **Feature Extraction** mode.

In [ ]:
for param in model.parameters():
    param.requires_grad = False

- CIFAR-10 dataset has 10 classes.
- Now only last layer will train.

In [ ]:
num_classes = 10
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

model.fc

Linear(in_features=512, out_features=10, bias=True)

Original ResNet was trained for 1000 ImageNet classes.

CIFAR-10 has 10 classes.

So we replace **final fully connected layer**.

model.fc.in_features
→ Automatically gets input size of last layer.

New layer:

* Input: same feature size

* Output: 10 classes

Now only this layer is trainable.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [ ]:
model.train()

for epoch in range(1):
    running_loss = 0.0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch Loss: {running_loss/len(trainloader)}")

Epoch Loss: 1.1873638294637203


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 62.0


Let's now fine-tune the `resnet18` model's 'layer 4' -
- Unfreeze last residual block (layer 4)
- Now optimizer should include those parameters
- Now we are fine-tuning deeper layers

In [ ]:
for name, param in model.named_parameters():
    if "layer4" in name:
        param.requires_grad = True

In [ ]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [ ]:
model.train()

for epoch in range(1):
    running_loss = 0.0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch Loss: {running_loss/len(trainloader)}")

Epoch Loss: 0.9161701668053865


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 75.33333333333333


Note: If you train from scratch (resnet18(pretrained=False)) -> accuracy will tend to increase more. Try it out!